<a href="https://colab.research.google.com/github/Timbo3399/INFM_SOTA2/blob/main/notebooks/workbook_valentin.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# All kinds of stuff :)

**Autor:** Valentin · **Datum:** 31.10.2026

# GPT2 :)

# Basic Attention Unit

In [2]:
import torch

In [3]:
inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

In [4]:
query = inputs[1] # [0.55, 0.87, 0.66], # journey  (x^2)

attention_score_2 = torch.empty(inputs.shape[0])
for i, x_i in enumerate(inputs):
  attention_score_2[i] = torch.dot(x_i, query)  # w'_i2 (Omege' index i j=2)

attention_score_2
# 0.9544 = 0.43, 0.15, 0.89] dot [0.55, 0.87, 0.66]

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])

In [5]:
attention_weights_2 = torch.softmax(attention_score_2, dim=0) # w_i2
attention_weights_2

tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])

In [6]:
context_vec_2 = torch.zeros(query.shape)
for i,x_i in enumerate(inputs):
    context_vec_2 += attention_weights_2[i]*x_i

context_vec_2

tensor([0.4419, 0.6515, 0.5683])

compute attentinon for all vectors

In [7]:
attention_score = torch.matmul(inputs, inputs.T)
attention_weights = torch.softmax(attention_score, dim=-1)
all_context_vectors = torch.matmul(attention_weights, inputs)
all_context_vectors

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])

trainable weights

In [8]:
x_2 = inputs[1]
d_in = inputs.shape[1]
d_out = 2 # simplification

torch.manual_seed(123)

# requires_grad=False zum testen auf False, fürs spätere trainieren sollten sie auf True
W_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_key   = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

In [10]:
query_2 = torch.matmul(x_2, W_query)
keys   = torch.matmul(inputs, W_key)
values = torch.matmul(inputs, W_value)

print(query_2)
print(keys.shape)


tensor([0.4306, 1.4551])
torch.Size([6, 2])


In [14]:
attention_score_2 = torch.matmul(query_2, keys.T)
d_k = keys.shape[1]
attention_weights_2 = torch.softmax(attention_score_2/d_k**0.5, dim=-1)

context_vec_2 = torch.matmul(attention_weights_2, values)
context_vec_2


tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820])

# Dataloader

## Result

In [ ]:
text = "Lorem ipsum dolor sit amet, consetetur sadipscing elitr, sed diam nonumy eirmod tempor invidunt ut labore et dolore magna aliquyam erat, sed diam voluptua. At vero eos et accusam et justo duo dolores et ea rebum. Stet clita kasd gubergren, no sea takimata sanctus est Lorem ipsum dolor sit amet. Lorem ipsum dolor sit amet, consetetur sadipscing elitr, sed diam nonumy eirmod tempor invidunt ut labore et dolore magna aliquyam erat, sed diam voluptua. At vero eos et accusam et justo duo dolores et ea rebum. Stet clita kasd gubergren, no sea takimata sanctus est Lorem ipsum dolor sit amet."

contextWindow = 4
output_dim = 128
vocab_size = 50257

In [ ]:
import tiktoken
import torch
from torch.utils.data import Dataset, DataLoader

In [ ]:
# Dataset and Datalaoder

class ContextWindowDataset(Dataset):
  def __init__(self, text:str, tokenizer, contextWindow:int, stride:int) -> None:
    self.inputIds = []
    self.targetIds = []

    # tokenize
    tokenIds = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
    assert len(tokenIds) > contextWindow, "Text is too short"

    for i in range(0, len(tokenIds)-contextWindow, stride):
      input = tokenIds[i:i+contextWindow]
      target = tokenIds[i+1:i+contextWindow+1]
      self.inputIds.append(torch.tensor(input))
      self.targetIds.append(torch.tensor(target))

  def __len__(self):
    return len(self.inputIds)

  def __getitem__(self, idx):
    return self.inputIds[idx], self.targetIds[idx]

In [ ]:
def createDataloader(text, contextWindow=4, stride=4, batch_size=8, drop_last=True, shuffle=False):
  tokenizer = tiktoken.get_encoding("gpt2")
  dataset = ContextWindowDataset(text, tokenizer, contextWindow, stride)
  return DataLoader(dataset, batch_size=batch_size, drop_last=drop_last, shuffle=shuffle)

In [ ]:
dataloader = createDataloader(text)

input, target = next(iter(dataloader))

In [ ]:
positoinEmbeddingLayer = torch.nn.Embedding(contextWindow, output_dim)
tokenEmbedingLayer = torch.nn.Embedding(vocab_size, output_dim)

embeddedToken = tokenEmbedingLayer(input)
embeddedPosition = positoinEmbeddingLayer(torch.arange(contextWindow))

embeddedImput = embeddedToken + embeddedPosition

print(embeddedImput.shape)# batch_size=8, contextWindow=4, output_dim = 128

torch.Size([8, 4, 128])


## Tesing and Understanding

In [ ]:
text = "Once upon a time there was a lovely princess. But she had an enchantment upon her of a fearful sort, which could only be broken by Love's first kiss. She was locked away in a castle guarded by a terrible fire breathing dragon. Many brave knights had attempted to free her from this dreadful prison, but none prevailed. She waited in the dragon's keep in the highest room of the tallest tower for her true love and true love's first kiss. Like that's ever going to happen. What a loony. Shrek Beware Stay out I think he's in here."

In [ ]:
import tiktoken
import re
contextWindow = 4

tokenizer = tiktoken.get_encoding("gpt2")

# Convert Text to tokens(integers)
tokenId = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(tokenId)

In [ ]:
# Seeing the decoded tokens
integersDecoded = "";
for i in tokenId:
  token = tokenizer.decode([i])
  if token[0] == " ":
    integersDecoded += token
  elif re.match(r"[.,!?;:]", token):
    integersDecoded += token
  else:
    integersDecoded += "-" + token

print(integersDecoded)

In [ ]:
import torch
torch.manual_seed(123)
# Sliding window
if len(tokenId) < contextWindow:
  contextWindow = len(tokenId)-1

stride = contextWindow
input_id = []
target_id = []

for i in range(0, len(tokenId)-contextWindow, stride):
  input = tokenId[i:i+contextWindow]
  target = tokenId[i+1:i+contextWindow+1]
  input_id.append(torch.tensor(input))
  target_id.append(torch.tensor(target))

print(input_id)
print(target_id)

In [ ]:
# Dataset and Datalaoder
from torch.utils.data import Dataset, DataLoader

class ContextWindowDataset(Dataset):
  def __init__(self, text:str, tokenizer, contextWindow:int, stride:int) -> None:
    self.inputIds = []
    self.targetIds = []

    # tokenize
    tokenIds = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
    assert len(tokenIds) > contextWindow, "Text is too short"

    for i in range(0, len(tokenId)-contextWindow, stride):
      input = tokenIds[i:i+contextWindow]
      target = tokenIds[i+1:i+contextWindow+1]
      self.inputIds.append(torch.tensor(input))
      self.targetIds.append(torch.tensor(target))

  def __len__(self):
    return len(self.inputIds)

  def __getitem__(self, idx):
    return self.inputIds[idx], self.targetIds[idx]

In [ ]:
def createDataloader(text, contextWindow=4, stride=1, batch_size=8, drop_last=True, shuffle=False):
  tokenizer = tiktoken.get_encoding("gpt2")
  dataset = ContextWindowDataset(text, tokenizer, contextWindow, stride)
  return DataLoader(dataset, batch_size=batch_size, drop_last=drop_last, shuffle=shuffle)

In [ ]:
dataloader = createDataloader(text)

input, target = next(iter(dataloader))
print(input)
print(target)

print(f'input shape: {input.shape}')
print(f'target shape: {target.shape}') #batches = 8, ContextWindow = 4

In [ ]:
# Embedding for Tokens
vocab_size = tokenizer.n_vocab #50257
output_dim = 16

tokenEmbedingLayer = torch.nn.Embedding(vocab_size, output_dim)
print(tokenEmbedingLayer) #vocabular=50257, Embedding-Dimensions=16

In [ ]:
# Embedding for Position
positoinEmbeddingLayer = torch.nn.Embedding(contextWindow, output_dim)
print(positoinEmbeddingLayer) # ContextWindow= 4, Output-Dimentions=16

In [ ]:
embeddedToken = tokenEmbedingLayer(input)
embeddedPosition = positoinEmbeddingLayer(torch.arange(contextWindow))

print(f'embedded_token shape: {embeddedToken.shape}')
print(f'embedded_position shape: {embeddedPosition.shape}')

In [ ]:
embeddedImput = embeddedToken + embeddedPosition
print(embeddedImput.shape)